In [ ]:
# === Merge (SUM) without any cap; remove overlap Neighbors-TopK ===
import pickle
from collections import defaultdict
from google.colab import drive

# === 0. Mount Google Drive ===
drive.mount('/content/drive')
topk_paths = [
    "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gcs_task1CS_cifar10_topk.pkl",
    "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gcs_task2CS_cifar10_topk.pkl",
]
neigh_paths = [
    "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gcs_task1CS_cifar10_neighbors.pkl",
    "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gcs_task2CS_cifar10_neighbors.pkl",
]

out_topk_path  = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gcsfisher_task1_2CS_cifar10_topk.pkl"
out_neigh_path = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gcsfisher_task1_2CS_cifar10_neighbors.pkl"

def load_list(p):
    with open(p, "rb") as f:
        return pickle.load(f)

# 1) Merge Top-K entries by summing CS values
topk_union = defaultdict(lambda: {"name": None, "index": None, "cs": 0.0})
for p in topk_paths:
    for e in load_list(p):
        key = (e["name"], int(e["index"]))
        if topk_union[key]["name"] is None:
            topk_union[key]["name"]  = e["name"]
            topk_union[key]["index"] = int(e["index"])
        topk_union[key]["cs"] += float(e.get("cs", 0.0))

topk_merged = list(topk_union.values())
topk_keys   = {(d["name"], d["index"]) for d in topk_merged}

# 2) Merge Neighbors by summing CS values and remove any overlap with Top-K
neigh_union = defaultdict(lambda: {"name": None, "index": None, "position": (), "cs": 0.0})
for p in neigh_paths:
    for it in load_list(p):
        key = (it["name"], int(it["index"]))
        if key in topk_keys:
            continue  # Prevent an entry from being both a neighbor and a Top-K weight
        if neigh_union[key]["name"] is None:
            neigh_union[key]["name"]  = it["name"]
            neigh_union[key]["index"] = int(it["index"])
            if "position" in it:
                neigh_union[key]["position"] = tuple(it["position"])
        neigh_union[key]["cs"] += float(it.get("cs", 0.0))

neighbors_merged = list(neigh_union.values())

# 3) Final check: intersection must be zero
inter = sum(1 for n in neighbors_merged if (n["name"], n["index"]) in topk_keys)
print("Intersection(Neighbors, TopK) =", inter)

# 4) Save merged results without truncation
with open(out_topk_path,  "wb") as f: pickle.dump(topk_merged, f)
with open(out_neigh_path, "wb") as f: pickle.dump(neighbors_merged, f)

print(f"TopK merged: {len(topk_merged)} | Neighbors merged: {len(neighbors_merged)}")


Mounted at /content/drive
Intersection(Neighbors, TopK) = 0
TopK merged: 31343 | Neighbors merged: 17231
